In [6]:
pip install chromadb openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ------------------------------------------------------------
# BLOCK 1: Import the ChromaDB library
#
# Same as "import openai" in your ESR1 agent
# Must be run before any other block — loads the library
# into memory so all cells below can use it
# ------------------------------------------------------------
import chromadb

In [3]:
# ------------------------------------------------------------
# BLOCK 2: Open a connection to ChromaDB
#
# PersistentClient = data saves to disk between runs
# (opposite of in-memory which forgets when Python closes)
#
# path="chroma_db" = the folder where ChromaDB stores data
# This folder does NOT exist yet — ChromaDB creates it
# automatically the first time this script runs
#
# SQL analogy:
#   SQL Server: conn = pyodbc.connect("server=...; database=...")
#   ChromaDB:   client = chromadb.PersistentClient(path="...")
# Both lines mean: "connect me to my database"
# ------------------------------------------------------------
client = chromadb.PersistentClient(path="chroma_db")


In [4]:
# ------------------------------------------------------------
# BLOCK 3: Create a collection
#
# A collection = a table in SQL Server, but for vectors
# We name ours "rwe_literature" — our persistent paper library
#
# get_or_create_collection means:
#   - First time running → creates the collection fresh
#   - Every time after → opens the existing one safely
#
# This prevents crashes if you run the script multiple times
#
# SQL analogy:
#   SQL Server: CREATE TABLE IF NOT EXISTS rwe_literature
#   ChromaDB:   get_or_create_collection(name="rwe_literature")
#   Both mean: "create this table only if it doesn't exist yet"
# ------------------------------------------------------------
collection = client.get_or_create_collection(name="rwe_literature")

In [5]:
# ------------------------------------------------------------
# BLOCK 4: Store the 5 paper abstracts into ChromaDB
#
# We are giving ChromaDB three things for each paper:
#   1. documents — the actual text of the abstract
#   2. ids       — unique identifier, like a primary key in SQL
#   3. metadatas — extra labels for filtering later (topic, disease)
#
# ChromaDB automatically converts each document into a vector
# using its built-in embedder (all-MiniLM-L6-v2) — we do not
# write any embedding code ourselves in this step
#
# WHY FAKE ABSTRACTS:
# Real PubMed calls would add complexity — if something broke
# we would not know if the bug is in RAG or PubMed code.
# Fake abstracts isolate ChromaDB as the only moving part.
# We swap these for real PubMed results in a later step.
#
# SQL analogy:
#   SQL Server: INSERT INTO rwe_literature VALUES (...)
#   ChromaDB:   collection.add(documents=..., ids=..., metadatas=...)
#   Both mean:  "put this data into my table"
# ------------------------------------------------------------

documents = [
    # paper_001: based on your Flatiron ESR1 mutation study
    "ESR1 mutation positivity was observed in 21 percent of metastatic breast cancer patients. "
    "Patients with ESR1 mutations showed reduced sensitivity to aromatase inhibitors.",

    # paper_002: based on your Optum NSCLC HCRU study
    "IPTW-adjusted analysis of NSCLC patients showed targeted therapy was associated with "
    "lower HCRU compared to immunotherapy plus chemotherapy combination regimens.",

    # paper_003: based on your VBA K-Means and XGBoost work
    "K-Means clustering identified three adherence phenotypes in VBA patients: persistent, "
    "early dropout, and gradual decliner. XGBoost predicted dropout with AUC 0.81.",

    # paper_004: based on your ESR1 biomarker testing patterns work
    "Flatiron Health real-world data showed ESR1 biomarker testing rates increased after "
    "FDA approval of elacestrant. Testing patterns varied by practice setting.",

    # paper_005: based on your survival analysis work in mBC
    "Cox proportional hazards model estimated time to treatment discontinuation in ER-positive "
    "HER2-negative metastatic breast cancer. Median TTD was 11.2 months on CDK4/6 inhibitors."
]

# unique ID for each paper — like a primary key in SQL Server
# if you try to add the same ID twice, ChromaDB raises an error
# this protects your database from storing duplicate papers
ids = ["paper_001", "paper_002", "paper_003", "paper_004", "paper_005"]

# metadata — extra labels for filtering
# topic = what the paper is about
# disease = which disease area it covers
# used when you want meaning search + hard filter combined
metadatas = [
    {"topic": "ESR1",      "disease": "mBC"},
    {"topic": "HCRU",      "disease": "NSCLC"},
    {"topic": "adherence", "disease": "VBA"},
    {"topic": "ESR1",      "disease": "mBC"},
    {"topic": "TTD",       "disease": "mBC"}
]

# add everything to the collection
# ChromaDB embeds the text automatically — no embedding code needed here
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

# confirm how many documents are now stored
print("✓ Documents stored:", collection.count())

C:\Users\bansa\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:10<00:00, 8.22MiB/s]


✓ Documents stored: 5


In [9]:
# ------------------------------------------------------------
# BLOCK 5: Query the ChromaDB collection
#
# We ask ChromaDB a question in plain English
# It converts our question into a vector automatically
# Then it measures the distance between our question vector
# and every stored paper vector
# The closest papers — lowest distance — come back as results
#
# n_results = how many results to return (like TOP 2 in SQL)
#
# SQL analogy:
#   SQL Server: SELECT TOP 2 * FROM rwe_literature
#               ORDER BY similarity DESC
#   ChromaDB:   collection.query(query_texts=[...], n_results=2)
#   Both mean:  "find me the 2 most relevant papers"
# ------------------------------------------------------------

results = collection.query(
    query_texts=["ESR1 biomarker testing in breast cancer"], # our question in plain English
    n_results=2                                               # return top 2 most relevant papers
)

# ------------------------------------------------------------
# Print the results clearly
# results is a dictionary with three keys we care about:
#   documents — the actual abstract text
#   distances — how similar each result is (lower = better)
#   metadatas — the topic and disease labels
# ------------------------------------------------------------

print("TOP 2 MOST RELEVANT PAPERS:")
print("=" * 50)

# loop through the 2 results and print each one
# results["documents"][0] is a list of the returned documents
# we use enumerate to get both the position number and the value
for i, (doc, distance, meta) in enumerate(zip(
    results["documents"][0],   # the abstract texts
    results["distances"][0],   # the distance scores
    results["metadatas"][0]    # the metadata labels
)):
    print(f"\nResult {i+1}:")
    print(f"  Distance : {distance:.4f}")        # 4 decimal places
    print(f"  Topic    : {meta['topic']}")        # from our metadata
    print(f"  Disease  : {meta['disease']}")      # from our metadata
    print(f"  Abstract : {doc[:120]}...")          # first 120 characters only
    print("-" * 50)

TOP 2 MOST RELEVANT PAPERS:

Result 1:
  Distance : 0.5026
  Topic    : ESR1
  Disease  : mBC
  Abstract : ESR1 mutation positivity was observed in 21 percent of metastatic breast cancer patients. Patients with ESR1 mutations s...
--------------------------------------------------

Result 2:
  Distance : 0.7860
  Topic    : ESR1
  Disease  : mBC
  Abstract : Flatiron Health real-world data showed ESR1 biomarker testing rates increased after FDA approval of elacestrant. Testing...
--------------------------------------------------


In [10]:
# ------------------------------------------------------------
# BLOCK 6: Query with a metadata filter
#
# We combine meaning search WITH a hard filter
# ChromaDB first narrows the pool to only mBC papers
# then runs similarity search within that smaller pool
#
# This is like SQL:
#   SELECT TOP 2 * FROM rwe_literature
#   WHERE disease = 'mBC'
#   ORDER BY similarity
#
# $eq means "equals" — ChromaDB's syntax for filtering
# Other operators: $ne (not equal), $in (in a list)
#
# IMPORTANT: n_results cannot exceed the number of papers
# that survive the filter — we have 3 mBC papers so
# n_results=2 is safe here
# ------------------------------------------------------------

results_filtered = collection.query(
    query_texts=["ESR1 biomarker testing in breast cancer"],  # our question
    where={"disease": {"$eq": "mBC"}},                       # only search mBC papers
    n_results=2                                               # return top 2
)

# ------------------------------------------------------------
# Print results — same structure as Block 5
# We expect only mBC papers to appear — NSCLC and VBA
# papers should be completely absent from results
# ------------------------------------------------------------

print("TOP 2 MOST RELEVANT mBC PAPERS ONLY:")
print("=" * 50)

for i, (doc, distance, meta) in enumerate(zip(
    results_filtered["documents"][0],
    results_filtered["distances"][0],
    results_filtered["metadatas"][0]
)):
    print(f"\nResult {i+1}:")
    print(f"  Distance : {distance:.4f}")
    print(f"  Topic    : {meta['topic']}")
    print(f"  Disease  : {meta['disease']}")
    print(f"  Abstract : {doc[:120]}...")
    print("-" * 50)

TOP 2 MOST RELEVANT mBC PAPERS ONLY:

Result 1:
  Distance : 0.5026
  Topic    : ESR1
  Disease  : mBC
  Abstract : ESR1 mutation positivity was observed in 21 percent of metastatic breast cancer patients. Patients with ESR1 mutations s...
--------------------------------------------------

Result 2:
  Distance : 0.7860
  Topic    : ESR1
  Disease  : mBC
  Abstract : Flatiron Health real-world data showed ESR1 biomarker testing rates increased after FDA approval of elacestrant. Testing...
--------------------------------------------------


In [11]:
# ------------------------------------------------------------
# BLOCK 7: Apply a distance threshold to filter weak results
#
# ChromaDB always returns results — it has no built-in concept
# of "this result is too irrelevant to show"
# We apply the threshold ourselves after getting results back
#
# THRESHOLD GUIDE for ChromaDB's built-in embedder:
#   below 0.6  → strong match, definitely relevant
#   0.6 – 0.9  → moderate match, probably relevant
#   above 0.9  → weak match, likely irrelevant
#   above 1.2  → almost certainly irrelevant
#
# We use 0.9 as our cutoff — anything above is discarded
# ------------------------------------------------------------

# our distance threshold — anything above this gets discarded
THRESHOLD = 0.9

# ------------------------------------------------------------
# Run two queries to demonstrate the threshold in action:
#   Query A — relevant query → papers should pass threshold
#   Query B — irrelevant query → papers should fail threshold
# ------------------------------------------------------------

# Query A: relevant — about ESR1 in breast cancer
# we expect paper_001 and paper_004 to pass the threshold
print("QUERY A: ESR1 biomarker testing in breast cancer")
print("=" * 50)

results_a = collection.query(
    query_texts=["ESR1 biomarker testing in breast cancer"],
    n_results=5  # get all 5 so we can see full threshold effect
)

# loop through results and apply threshold manually
# we check each distance — if above THRESHOLD we discard it
relevant_a = []  # empty list to collect papers that pass

for doc, distance, meta in zip(
    results_a["documents"][0],
    results_a["distances"][0],
    results_a["metadatas"][0]
):
    if distance < THRESHOLD:  # only keep results below threshold
        relevant_a.append((doc, distance, meta))  # add to our list

# now print whatever passed the threshold
if len(relevant_a) == 0:
    # nothing passed — this is where we would fall back to PubMed
    print("No relevant papers found in memory store.")
    print("→ In production: fall back to live PubMed search")
else:
    for i, (doc, distance, meta) in enumerate(relevant_a):
        print(f"\nResult {i+1} (passed threshold):")
        print(f"  Distance : {distance:.4f}")
        print(f"  Topic    : {meta['topic']}")
        print(f"  Disease  : {meta['disease']}")
        print(f"  Abstract : {doc[:120]}...")
        print("-" * 50)

print(f"\n{len(relevant_a)} of 5 papers passed the threshold of {THRESHOLD}")

# ------------------------------------------------------------
# Query B: irrelevant — nothing to do with our RWE papers
# we expect ALL results to fail the threshold
# ------------------------------------------------------------
print("\n\nQUERY B: what is the weather forecast for Delhi")
print("=" * 50)

results_b = collection.query(
    query_texts=["what is the weather forecast for Delhi"],
    n_results=5
)

relevant_b = []  # empty list to collect papers that pass

for doc, distance, meta in zip(
    results_b["documents"][0],
    results_b["distances"][0],
    results_b["metadatas"][0]
):
    if distance < THRESHOLD:  # only keep results below threshold
        relevant_b.append((doc, distance, meta))

if len(relevant_b) == 0:
    # nothing passed — correct behaviour for an irrelevant query
    print("No relevant papers found in memory store.")
    print("→ In production: fall back to live PubMed search")
else:
    for i, (doc, distance, meta) in enumerate(relevant_b):
        print(f"\nResult {i+1} (passed threshold):")
        print(f"  Distance : {distance:.4f}")
        print(f"  Topic    : {meta['topic']}")
        print(f"  Disease  : {meta['disease']}")
        print(f"  Abstract : {doc[:120]}...")
        print("-" * 50)

print(f"\n{len(relevant_b)} of 5 papers passed the threshold of {THRESHOLD}")

QUERY A: ESR1 biomarker testing in breast cancer

Result 1 (passed threshold):
  Distance : 0.5026
  Topic    : ESR1
  Disease  : mBC
  Abstract : ESR1 mutation positivity was observed in 21 percent of metastatic breast cancer patients. Patients with ESR1 mutations s...
--------------------------------------------------

Result 2 (passed threshold):
  Distance : 0.7860
  Topic    : ESR1
  Disease  : mBC
  Abstract : Flatiron Health real-world data showed ESR1 biomarker testing rates increased after FDA approval of elacestrant. Testing...
--------------------------------------------------

2 of 5 papers passed the threshold of 0.9


QUERY B: what is the weather forecast for Delhi
No relevant papers found in memory store.
→ In production: fall back to live PubMed search

0 of 5 papers passed the threshold of 0.9
